<a href="https://colab.research.google.com/github/unatisaini/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unatisaini/flyrank-internship-mi/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (page, query) pair for a given month. Time window: monthly grain, using month=2026-03 as the working/test month (per the warehouse's monthly partitioning).

In [3]:
# Verify grain: one row per (page, query, month)
query = """
SELECT page, query, month, COUNT(*) as row_count
FROM `your_dataset.search_performance`
WHERE month = '2026-03'
GROUP BY page, query, month
HAVING COUNT(*) > 1
"""
# Should return 0 rows if grain is correct


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: avg_position, impressions_30d, query_length, page_type, device_category
Label/proxy: CTR opportunity score (impressions high, CTR below expected-for-position)
Context (not features, just identifiers): page URL, query text, month
Excluded: rows with impressions < 10 (too noisy to trust CTR on) — excluded because low-volume CTR is statistically unreliable

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# Check 1: row count + date span
q1 = """
SELECT COUNT(*) as total_rows, MIN(month) as first_month, MAX(month) as last_month
FROM `your_dataset.search_performance`
"""

# Check 2: availability filter
q2 = """
SELECT COUNT(*) as available_rows
FROM `your_dataset.search_performance`
WHERE month = '2026-03' AND available IS TRUE
"""

# Check 3: missing values check
q3 = """
SELECT
  COUNTIF(position IS NULL) as missing_position,
  COUNTIF(impressions IS NULL) as missing_impressions
FROM `your_dataset.search_performance`
WHERE month = '2026-03'
"""

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell you about pages with GSC-only history and no matching GA session data (early months before GA integration) — those rows will show impressions/position but null engagement fields. It also can't distinguish real CTR drops from seasonal query volume shifts, since we only have one month's window per row.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.